# BÀI THỰC HÀNH 4: MẠNG NEURAL HỒI QUY

MSSV: 23520078  
Họ tên: Trần Nhật Phương Anh

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses.

Bộ dữ liệu sử dụng: [PhoMT](https://drive.google.com/drive/folders/186OAOuSEYEDVcry7WP5UBdqECXo26QAb?usp=drive_link).

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import unicodedata
from underthesea import word_tokenize as vi_tokenize
import re
import nltk
from nltk.tokenize import word_tokenize as en_tokenize
from tqdm import tqdm
from collections import Counter
from rouge import Rouge

nltk.download('punkt')

[nltk_data] Downloading package punkt to C:\Users\Phuong
[nltk_data]     Anh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
train = pd.read_json('small-PhoMT/small-train.json')
dev = pd.read_json('small-PhoMT/small-dev.json')
test = pd.read_json('small-PhoMT/small-test.json')

In [3]:
train

,english,vietnamese
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ..."
...,...,...
19995,And the man was incredibly curious .,Và người đàn ông này cực kỳ tò mò .
19996,And he wanted to understand what it was and wh...,Và ông muốn hiểu nó là gì và tại sao thế rằng ...
19997,"And one day , we were walking .","Một ngày nọ , chúng tôi đang đi bộ ."
19998,"We were in France , in Les Houches .","Chúng tôi ở Pháp , tại Les Houches ."


In [4]:
dev

,english,vietnamese
0,"﻿Hurricane Dorian , one of the most powerful s...","Vào chủ nhật ngày 1-9-2019 , cơn bão Dorian , ..."
1,Dorian is especially dangerous due to its slow...,Bão Dorian đặc biệt nguy hiểm vì nó di chuyển ...
2,"The storm passed by the Leeward Islands , Puer...","Khi đi qua quần đảo Leeward , Puerto Rico và q..."
3,The United States branch office continues to g...,Văn phòng chi nhánh Hoa Kỳ tiếp tục cập nhật t...
4,"At this time , there have been no reported inj...","Theo báo cáo đến thời điểm hiện tại , trong 46..."
...,...,...
1995,You could even imagine a version of this scena...,Một dị bản của viễn cảnh này là nơi mà mọi ngư...
1996,"Over the course of last year , open - source h...","Năm ngoái , các hacker chuyên về ổ cứng mã ngu..."
1997,"At the other end of the network , there 'd be ...","Ở đầu kia của mạng lưới , sẽ có dịch vụ giúp c..."
1998,"Now , you Web 2.0 folks in the audience know w...",Những ai ở đây thuộc thế hệ Web 2.0 sẽ hiểu tô...


In [5]:
test

,english,vietnamese
0,"Brother Albert Barnett and his wife , Sister S...","Anh Albert Barnett và chị Susan Barnett , thuộ..."
1,Severe storms ripped through parts of the sout...,"Ngày 11 và 12-1-2020 , những cơn bão lớn đã qu..."
2,"Two days of heavy rain , high winds , and nume...",Những trận mưa to và gió lớn trong suốt hai ng...
3,"Sadly , Brother Albert Barnett and his wife , ...","Đáng buồn là anh Albert Barnett 85 tuổi , và v..."
4,The United States branch also reports that at ...,Chi nhánh Hoa Kỳ cũng cho biết có ít nhất bốn ...
...,...,...
1995,Toyota applied the principles of modularity of...,Toyota áp dụng các nguyên tắc của tính đơn lẻ ...
1996,"Now fortunately , few companies succumb to cat...",Thật may là một vài công ti không chống cự ngọ...
1997,But we do read in the newspaper every day abou...,Nhưng chúng ta đọc báo chí mỗi ngày về các côn...
1998,"How is it , then , that the consumer optics gi...",Sau đó thì gã khổng lồ tiêu dùng quang học có ...


In [6]:
print("Kích thước tập train", len(train))
print("Kích thước tập dev:", len(dev))
print("Kích thước tập test:", len(test))

Kích thước tập train 20000
Kích thước tập dev: 2000
Kích thước tập test: 2000


In [7]:
print('Các cột trong tập train:', train.columns.tolist())
print('Các cột trong tập dev:', dev.columns.tolist())
print('Các cột trong tập test:', test.columns.tolist())

Các cột trong tập train: ['english', 'vietnamese']
Các cột trong tập dev: ['english', 'vietnamese']
Các cột trong tập test: ['english', 'vietnamese']


# Data Preparation

## Chuẩn hóa text

In [8]:
def preprocess_text(text, language):
    # Chuẩn hóa Unicode
    text = unicodedata.normalize('NFC', text)
    
    # Chuyển về chữ thường
    text = text.lower()
    
    # Xóa các ký tự đặc biệt không cần thiết
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s\.\,\!\?]", "", text)
    
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    if language == "en":
        tokens = en_tokenize(text)   
    elif language == "vi":
        tokens = vi_tokenize(text)
        
    return tokens

In [9]:
for dataset in [train, dev, test]:
    tqdm.pandas()
    dataset['en_tokens'] = dataset['english'].progress_apply(lambda x: preprocess_text(x, "en"))
    dataset['vi_tokens'] = dataset['vietnamese'].progress_apply(lambda x: preprocess_text(x, "vi"))

100%|██████████| 2000/2000 [00:12<00:00, 160.75it/s]


In [10]:
train

,english,vietnamese,en_tokens,vi_tokens
0,It begins with a countdown .,Câu chuyện bắt đầu với buổi lễ đếm ngược .,"[it, begins, with, a, countdown, .]","[câu chuyện, bắt đầu, với, buổi, lễ, đếm, ngượ..."
1,"On August 14th , 1947 , a woman in Bombay goes...","Ngày 14 , tháng 8 , năm 1947 , gần nửa đêm , ở...","[on, august, 14th, ,, 1947, ,, a, woman, in, b...","[ngày, 14, ,, tháng, 8, ,, năm, 1947, ,, gần, ..."
2,"Across India , people hold their breath for th...","Cùng lúc , trên khắp đất Ấn , người ta nín thở...","[across, india, ,, people, hold, their, breath...","[cùng, lúc, ,, trên, khắp, đất, ấn, ,, người, ..."
3,"And at the stroke of midnight , a squirming in...","Khi đồng hồ điểm thời khắc nửa đêm , một đứa t...","[and, at, the, stroke, of, midnight, ,, a, squ...","[khi, đồng hồ, điểm, thời khắc, nửa đêm, ,, mộ..."
4,"These events form the foundation of "" Midnight...","Những sự kiện này là nền móng tạo nên "" Những ...","[these, events, form, the, foundation, of, mid...","[những, sự kiện, này, là, nền móng, tạo, nên, ..."
...,...,...,...,...
19995,And the man was incredibly curious .,Và người đàn ông này cực kỳ tò mò .,"[and, the, man, was, incredibly, curious, .]","[và, người, đàn ông, này, cực kỳ, tò mò, .]"
19996,And he wanted to understand what it was and wh...,Và ông muốn hiểu nó là gì và tại sao thế rằng ...,"[and, he, wanted, to, understand, what, it, wa...","[và, ông, muốn, hiểu, nó, là, gì, và, tại sao,..."
19997,"And one day , we were walking .","Một ngày nọ , chúng tôi đang đi bộ .","[and, one, day, ,, we, were, walking, .]","[một, ngày, nọ, ,, chúng tôi, đang, đi, bộ, .]"
19998,"We were in France , in Les Houches .","Chúng tôi ở Pháp , tại Les Houches .","[we, were, in, france, ,, in, les, houches, .]","[chúng tôi, ở, pháp, ,, tại, les, houches, .]"


In [11]:
dev

,english,vietnamese,en_tokens,vi_tokens
0,"﻿Hurricane Dorian , one of the most powerful s...","Vào chủ nhật ngày 1-9-2019 , cơn bão Dorian , ...","[hurricane, dorian, ,, one, of, the, most, pow...","[vào, chủ nhật, ngày, 192019, ,, cơn, bão, dor..."
1,Dorian is especially dangerous due to its slow...,Bão Dorian đặc biệt nguy hiểm vì nó di chuyển ...,"[dorian, is, especially, dangerous, due, to, i...","[bão, dorian, đặc biệt, nguy hiểm, vì, nó, di ..."
2,"The storm passed by the Leeward Islands , Puer...","Khi đi qua quần đảo Leeward , Puerto Rico và q...","[the, storm, passed, by, the, leeward, islands...","[khi, đi, qua, quần đảo, leeward, ,, puerto ri..."
3,The United States branch office continues to g...,Văn phòng chi nhánh Hoa Kỳ tiếp tục cập nhật t...,"[the, united, states, branch, office, continue...","[văn phòng, chi nhánh, hoa kỳ, tiếp tục, cập n..."
4,"At this time , there have been no reported inj...","Theo báo cáo đến thời điểm hiện tại , trong 46...","[at, this, time, ,, there, have, been, no, rep...","[theo, báo cáo, đến, thời điểm, hiện tại, ,, t..."
...,...,...,...,...
1995,You could even imagine a version of this scena...,Một dị bản của viễn cảnh này là nơi mà mọi ngư...,"[you, could, even, imagine, a, version, of, th...","[một, dị bản, của, viễn cảnh, này, là, nơi, mà..."
1996,"Over the course of last year , open - source h...","Năm ngoái , các hacker chuyên về ổ cứng mã ngu...","[over, the, course, of, last, year, ,, open, s...","[năm ngoái, ,, các, hacker, chuyên, về, ổ, cứn..."
1997,"At the other end of the network , there 'd be ...","Ở đầu kia của mạng lưới , sẽ có dịch vụ giúp c...","[at, the, other, end, of, the, network, ,, the...","[ở, đầu, kia, của, mạng lưới, ,, sẽ, có, dịch ..."
1998,"Now , you Web 2.0 folks in the audience know w...",Những ai ở đây thuộc thế hệ Web 2.0 sẽ hiểu tô...,"[now, ,, you, web, 2.0, folks, in, the, audien...","[những ai, ở, đây, thuộc, thế hệ, web 2.0, sẽ,..."


In [12]:
test

,english,vietnamese,en_tokens,vi_tokens
0,"Brother Albert Barnett and his wife , Sister S...","Anh Albert Barnett và chị Susan Barnett , thuộ...","[brother, albert, barnett, and, his, wife, ,, ...","[anh, albert, barnett, và, chị, susan, barnett..."
1,Severe storms ripped through parts of the sout...,"Ngày 11 và 12-1-2020 , những cơn bão lớn đã qu...","[severe, storms, ripped, through, parts, of, t...","[ngày, 11, và, 1212020, ,, những, cơn, bão, lớ..."
2,"Two days of heavy rain , high winds , and nume...",Những trận mưa to và gió lớn trong suốt hai ng...,"[two, days, of, heavy, rain, ,, high, winds, ,...","[những, trận, mưa, to, và, gió, lớn, trong, su..."
3,"Sadly , Brother Albert Barnett and his wife , ...","Đáng buồn là anh Albert Barnett 85 tuổi , và v...","[sadly, ,, brother, albert, barnett, and, his,...","[đáng, buồn, là, anh, albert, barnett, 85, tuổ..."
4,The United States branch also reports that at ...,Chi nhánh Hoa Kỳ cũng cho biết có ít nhất bốn ...,"[the, united, states, branch, also, reports, t...","[chi nhánh, hoa kỳ, cũng, cho, biết, có, ít nh..."
...,...,...,...,...
1995,Toyota applied the principles of modularity of...,Toyota áp dụng các nguyên tắc của tính đơn lẻ ...,"[toyota, applied, the, principles, of, modular...","[toyota, áp dụng, các, nguyên tắc, của, tính, ..."
1996,"Now fortunately , few companies succumb to cat...",Thật may là một vài công ti không chống cự ngọ...,"[now, fortunately, ,, few, companies, succumb,...","[thật, may, là, một vài, công ti, không, chống..."
1997,But we do read in the newspaper every day abou...,Nhưng chúng ta đọc báo chí mỗi ngày về các côn...,"[but, we, do, read, in, the, newspaper, every,...","[nhưng, chúng ta, đọc, báo chí, mỗi, ngày, về,..."
1998,"How is it , then , that the consumer optics gi...",Sau đó thì gã khổng lồ tiêu dùng quang học có ...,"[how, is, it, ,, then, ,, that, the, consumer,...","[sau, đó, thì, gã, khổng lồ, tiêu dùng, quang ..."


## Chọn độ dài padding

In [13]:
# Tính độ dài câu cho tiếng Anh và tiếng Việt
train['en_len'] = train['en_tokens'].apply(len)
train['vi_len'] = train['vi_tokens'].apply(len)
dev['en_len'] = dev['en_tokens'].apply(len)
dev['vi_len'] = dev['vi_tokens'].apply(len)
test['en_len'] = test['en_tokens'].apply(len)
test['vi_len'] = test['vi_tokens'].apply(len)

In [14]:
def ratio_under_threshold(df, col_name, threshold, name="Dataset"):
    total = len(df)
    count = (df[col_name] <= threshold).sum()
    ratio = count / total * 100
    print(f"Tỉ lệ câu {col_name} <= {threshold} từ trong {name}: {ratio:.2f}%")
    
for threshold in [50, 80, 100]:
    ratio_under_threshold(train, "en_len", threshold, "train-en")
    ratio_under_threshold(train, "vi_len", threshold, "train-vi")
    ratio_under_threshold(dev, "en_len", threshold, "dev-en")
    ratio_under_threshold(dev, "vi_len", threshold, "dev-vi")
    ratio_under_threshold(test, "en_len", threshold, "test-en")
    ratio_under_threshold(test, "vi_len", threshold, "test-vi")

Tỉ lệ câu en_len <= 50 từ trong train-en: 96.92%
Tỉ lệ câu vi_len <= 50 từ trong train-vi: 97.20%
Tỉ lệ câu en_len <= 50 từ trong dev-en: 98.85%
Tỉ lệ câu vi_len <= 50 từ trong dev-vi: 99.10%
Tỉ lệ câu en_len <= 50 từ trong test-en: 97.15%
Tỉ lệ câu vi_len <= 50 từ trong test-vi: 96.75%
Tỉ lệ câu en_len <= 80 từ trong train-en: 99.68%
Tỉ lệ câu vi_len <= 80 từ trong train-vi: 99.73%
Tỉ lệ câu en_len <= 80 từ trong dev-en: 99.95%
Tỉ lệ câu vi_len <= 80 từ trong dev-vi: 99.95%
Tỉ lệ câu en_len <= 80 từ trong test-en: 99.85%
Tỉ lệ câu vi_len <= 80 từ trong test-vi: 99.75%
Tỉ lệ câu en_len <= 100 từ trong train-en: 99.91%
Tỉ lệ câu vi_len <= 100 từ trong train-vi: 99.94%
Tỉ lệ câu en_len <= 100 từ trong dev-en: 99.95%
Tỉ lệ câu vi_len <= 100 từ trong dev-vi: 99.95%
Tỉ lệ câu en_len <= 100 từ trong test-en: 100.00%
Tỉ lệ câu vi_len <= 100 từ trong test-vi: 100.00%


Chọn max_len = 50 để tối ưu thời gian huấn luyện

## Xây dựng vocab

In [15]:
def build_vocab(token_list, min_freq = 2):
    counter = Counter([tok for seq in token_list for tok in seq])
    vocab = {}
    vocab['<pad>'] = 0
    vocab['<sos>'] = 1
    vocab['<eos>'] = 2
    vocab['<unk>'] = 3
    idx = 4
    for tok, freq in counter.items():
        if freq >= min_freq:
            vocab[tok] = idx
            idx += 1
    return vocab

en_vocab = build_vocab(train['en_tokens'].tolist(), min_freq = 2)
vi_vocab = build_vocab(train['vi_tokens'].tolist(), min_freq = 2) 

In [16]:
print("Kích thước từ điển tiếng Anh:", len(en_vocab))
print("Kích thước từ điển tiếng Việt:", len(vi_vocab))

Kích thước từ điển tiếng Anh: 10054
Kích thước từ điển tiếng Việt: 7644


## Chuyển token thành ID

In [17]:
def token_to_id(tokens, vocab, max_len = 50, add_sos_eos = False):
    ids = []
    if add_sos_eos:
        ids.append(vocab.get('<sos>'))
    for tok in tokens:
        ids.append(vocab.get(tok, vocab['<unk>']))
    if add_sos_eos:
        ids.append(vocab.get('<eos>'))
        
    if len(ids) < max_len:
        ids += [vocab.get('<pad>')] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    
    return ids

train['en_ids'] = train['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
train['vi_ids'] = train['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

dev['en_ids'] = dev['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
dev['vi_ids'] = dev['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

test['en_ids'] = test['en_tokens'].apply(lambda x: token_to_id(x, en_vocab, max_len=50, add_sos_eos=False))
test['vi_ids'] = test['vi_tokens'].apply(lambda x: token_to_id(x, vi_vocab, max_len=50, add_sos_eos=True))

## Tạo Dataset và DataLoader

In [18]:
class PhoMTDataset(Dataset):
    def __init__(self, src_ids, tgt_ids):
        self.src = src_ids
        self.tgt = tgt_ids
    def __len__(self):
        return len(self.src)
    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])
    
train_dataset = PhoMTDataset(train['en_ids'].tolist(), train['vi_ids'].tolist())
dev_dataset = PhoMTDataset(dev['en_ids'].tolist(), dev['vi_ids'].tolist())
test_dataset = PhoMTDataset(test['en_ids'].tolist(), test['vi_ids'].tolist())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Huấn luyện mô hình

In [19]:
def ids_to_text(seq, id2tok):
    return " ".join([id2tok[int(i)] for i in seq if int(i) in id2tok])

In [20]:
def train_model(model, train_loader, dev_loader, optimizer, loss_fn, num_epochs=20, patience=3, device="cuda"):
    id2tok = {idx: tok for tok, idx in vi_vocab.items()}
    best_rouge = 0.0
    counter = 0
    model.to(device)
    rouge = Rouge()
    
    for epoch in range(1, num_epochs+1):
        model.train()
        train_loss = 0
        for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch}"):
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            
            logits = model(src, tgt, teacher_forcing_ratio=0.5)  
            B, T, V = logits.size()
    
            loss = loss_fn(logits.reshape(B*T, V), tgt[:,1:].reshape(B*T))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        dev_loss = 0
        rouge_scores = []
        with torch.no_grad():
            for src, tgt in dev_loader:
                src, tgt = src.to(device), tgt.to(device)
                logits = model(src, tgt, teacher_forcing_ratio=0.0)  
                B, T, V = logits.size()
                loss = loss_fn(logits.reshape(B*T, V), tgt[:,1:].reshape(B*T))
                dev_loss += loss.item()
                
                pred_ids = torch.argmax(logits, dim=-1)
                pred_texts = [ids_to_text(seq, id2tok) for seq in pred_ids]
                tgt_texts = [ids_to_text(seq, id2tok) for seq in tgt]

                for p, t in zip(pred_texts, tgt_texts):
                    scores = rouge.get_scores(p, t)
                    rouge_scores.append(scores[0]["rouge-l"]["f"])

        dev_loss /= len(dev_loader)
        avg_rouge = sum(rouge_scores) / len(rouge_scores)

        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Dev Loss = {dev_loss:.4f}, Dev ROUGE-L = {avg_rouge:.4f}")

        if avg_rouge > best_rouge:
            best_rouge = avg_rouge
            counter = 0
            torch.save(model.state_dict(), "best_model3.pt")
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping ở epoch {epoch}")
                break

# Đánh giá mô hình

In [37]:
def translate_sentence(model, src, max_len=50, device="cuda"):
    model.eval()
    src = src.unsqueeze(0).to(device) 

    enc_out, (h, c) = model.encoder(src)
    h = model._sum_bidir(h)
    c = model._sum_bidir(c)

    src_mask = (src != model.pad_id).long()

    input_t = torch.tensor([[model.sos_id]], device=device)
    outputs = []
    prev_tilde_h = None

    for _ in range(max_len):
        logits, h, c, attn, prev_tilde_h = model.decoder(
            input_t, h, c, enc_out, src_mask=src_mask, prev_tilde_h=prev_tilde_h
        )
        next_tok = logits.argmax(dim=-1)  
        outputs.append(next_tok.item())
        input_t = next_tok.unsqueeze(1)
        if next_tok.item() == model.eos_id:
            break

    return outputs

In [38]:
def evaluate_rouge(model, data_loader, vi_vocab, device="cuda"):
    id2word = {idx: tok for tok, idx in vi_vocab.items()}
    rouge = Rouge()
    scores = []
    for src, tgt in data_loader:
        src, tgt = src.to(device), tgt.to(device)
        for i in range(src.size(0)):
            pred_ids = translate_sentence(model, src[i], max_len=50, device=device)
            pred_tokens = [id2word.get(idx, "<unk>") for idx in pred_ids]
            ref_tokens = [id2word.get(idx, "<unk>") for idx in tgt[i].cpu().numpy() if idx not in [vi_vocab['<pad>'], vi_vocab['<sos>'], vi_vocab['<eos>']]]
            pred_sent = " ".join(pred_tokens)
            ref_sent = " ".join(ref_tokens)
            score = rouge.get_scores(pred_sent, ref_sent)[0]['rouge-l']['f']
            scores.append(score)
    avg_score = sum(scores) / len(scores)
    print(f"ROUGE-L F1: {avg_score:.4f}")

#### Bài 3: Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [23]:
class LuongAttention(nn.Module):
    """
    Luong Attention (Multiplicative Attention)

    Công thức:
    score(s_t, h_i) = s_t^T * W * h_i
    """
    def __init__(self, hidden_size):
        super(LuongAttention, self).__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, decoder_hidden, encoder_outputs, src_mask=None):
        """
        Args:
            decoder_hidden: [batch_size, hidden_size]
            encoder_outputs: [batch_size, src_len, hidden_size]

        Returns:
            context: [batch_size, hidden_size]
            attention_weights: [batch_size, src_len]
        """
        # Luong: score = s_t^T * W * h_i
        query = decoder_hidden.unsqueeze(1)  # [batch_size, 1, hidden_size]
        keys = self.W(encoder_outputs)  # [batch_size, src_len, hidden_size]

        # Multiplicative attention
        scores = torch.bmm(query, keys.transpose(1, 2)).squeeze(1)  # [batch_size, src_len]

        if src_mask is not None:
            scores = scores.masked_fill(src_mask == 0, float('-inf'))

        # Softmax
        attention_weights = torch.softmax(scores, dim=1)

        # Weighted sum
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)

        return context, attention_weights

In [24]:
class EncoderAttention(nn.Module):
    def __init__(self, vocab_size, emb_size=256, hidden_size=256, num_layers=3, pad_id=0, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0,
                            bidirectional=True)
        self.out_proj = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, src, src_len=None):
        emb = self.embedding(src) 
        if src_len is not None:
            packed = nn.utils.rnn.pack_padded_sequence(emb, src_len.cpu(), batch_first=True, enforce_sorted=False)
            outputs, (h, c) = self.lstm(packed)
            outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True) 
        else:
            outputs, (h, c) = self.lstm(emb)  
        outputs = self.out_proj(outputs)      
        return outputs, (h, c)

In [25]:
class DecoderAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_size=256,
                 num_layers=3, dropout=0.3, pad_id=0, input_feeding=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.input_feeding = input_feeding
        rnn_inp = embedding_dim + (hidden_size if input_feeding else 0)
        self.lstm = nn.LSTM(rnn_inp, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.attention = LuongAttention(hidden_size)
        self.Wc = nn.Linear(2 * hidden_size, hidden_size)  
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs, src_mask=None, prev_tilde_h=None):
        emb = self.dropout(self.embedding(input_token))
        if self.input_feeding:
            if prev_tilde_h is None:
                prev_tilde_h = torch.zeros(input_token.size(0), self.lstm.hidden_size, device=input_token.device)
            x = torch.cat([emb, prev_tilde_h.unsqueeze(1)], dim=-1)
        else:
            x = emb

        out, (hidden, cell) = self.lstm(x, (hidden, cell))   
        dec_h = hidden[-1]                                   
        context, attn = self.attention(dec_h, encoder_outputs, src_mask)
        combined = torch.cat([out.squeeze(1), context], dim=1) 
        tilde_h = torch.tanh(self.Wc(combined))               
        logits = self.fc(self.dropout(tilde_h))              
        return logits, hidden, cell, attn, tilde_h

In [26]:
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder, sos_id, eos_id, pad_id):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.sos_id = sos_id
        self.eos_id = eos_id
        self.pad_id = pad_id

    def _sum_bidir(self, h):
        num_layers = h.size(0) // 2
        forward  = h[:num_layers]
        backward = h[num_layers:]
        return forward + backward 

    def forward(self, src, tgt, src_len=None, teacher_forcing_ratio=0.5):
        enc_out, (h, c) = self.encoder(src, src_len=src_len)     
        h = self._sum_bidir(h)
        c = self._sum_bidir(c)

        src_mask = (src != self.pad_id).long()                

        B, T = tgt.size()
        outputs = []
        input_t = tgt[:, 0].unsqueeze(1)                  
        prev_tilde_h = None

        for t in range(1, T):
            logits, h, c, attn, prev_tilde_h = self.decoder(
                input_t, h, c, enc_out, src_mask=src_mask, prev_tilde_h=prev_tilde_h
            )
            outputs.append(logits.unsqueeze(1))                  

            use_tf = torch.rand(B, device=tgt.device) < teacher_forcing_ratio
            greedy = logits.argmax(dim=-1)                      
            next_tok = torch.where(use_tf, tgt[:, t], greedy)
            input_t = next_tok.unsqueeze(1)

        return torch.cat(outputs, dim=1)                 

In [27]:
PAD_ID = 0
SOS_ID = 1
EOS_ID = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

enc = EncoderAttention(len(en_vocab), pad_id=PAD_ID)
dec = DecoderAttention(len(vi_vocab), pad_id=PAD_ID)
model = Seq2SeqAttention(enc, dec, SOS_ID, EOS_ID, PAD_ID).to(device)

optimizer = optim.Adam(model.parameters(), lr= 0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

In [28]:
train_model(model, train_loader, dev_loader, optimizer, loss_fn, num_epochs=20, patience=3, device=device)

Epoch 1: 100%|██████████| 625/625 [1:32:44<00:00,  8.90s/it]


Epoch 1: Train Loss = 6.2453, Dev Loss = 5.9833, Dev ROUGE-L = 0.1666


Epoch 2: 100%|██████████| 625/625 [1:27:24<00:00,  8.39s/it]


Epoch 2: Train Loss = 5.7410, Dev Loss = 5.7619, Dev ROUGE-L = 0.2423


Epoch 3: 100%|██████████| 625/625 [1:20:27<00:00,  7.72s/it]


...

Epoch 18: Train Loss = 3.6320, Dev Loss = 5.8014, Dev ROUGE-L = 0.3551


Epoch 19: 100%|██████████| 625/625 [26:25<00:00,  2.54s/it]


Epoch 19: Train Loss = 3.5562, Dev Loss = 5.8486, Dev ROUGE-L = 0.3548


Epoch 20: 100%|██████████| 625/625 [26:40<00:00,  2.56s/it]


Epoch 20: Train Loss = 3.4837, Dev Loss = 5.8648, Dev ROUGE-L = 0.3562


In [29]:
model.load_state_dict(torch.load("best_model3.pt", map_location=device))

<All keys matched successfully>

In [39]:
evaluate_rouge(model, test_loader, vi_vocab, device=device)

ROUGE-L F1: 0.3372
